In [17]:
%load_ext autoreload
%autoreload 2
from sindex.sources.datacite.utils import get_relevant_citations_block_from_ndjson
from sindex.sources.datacite.jobs import (
    batch_slim_datacite_record_to_ndjson_fast,
batch_find_citations_dc_from_citation_block_optimized,
extract_unique_dois_from_citation_blocks,
lookup_dates_in_oa_snapshot,
batch_find_citations_from_dc_parallel,
batch_slim_datacite_chunked
)
import duckdb
from pathlib import Path
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Slim datacite citations raw ndjson

In [ ]:
src_folder = r"D:\pipeline-data\citations\datacite\datacite-raw-with-citations"
dst_folder = r"D:\pipeline-data\citations\datacite\datacite-slim-with-citations"
summary = batch_slim_datacite_chunked(
    src_folder=src_folder,
    dst_folder=dst_folder)

Found 769 input files.
Starting processing with 32 cores. Batch size: 100,000...
Processed: 1,800,000 lines | Batches: 18 | Bad: 0

In [3]:
def count_lines(directory):
    total_lines = 0
    for filename in os.listdir(directory):
        if filename.endswith(".ndjson"):
            path = os.path.join(directory, filename)
            with open(path, 'rb') as f:
                count = sum(1 for line in f)
                total_lines += count
    return total_lines
slim_path = r"D:\pipeline-data\citations\datacite\datacite-slim-with-citations"
print(f"Total: {count_lines(slim_path)}")

Total: 48964382


## Save citations block matching to our DOIs

In [2]:
get_relevant_citations_block_from_ndjson(
     db_path = r"D:\pipeline-data\records\slim-records\datacite-slim-records.duckdb",
     ndjson_folder = r"D:\pipeline-data\citations\datacite\datacite-slim-with-citations",
     target_table = "my_datasets",
     output_file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson",
     reset_log = True
 )

[2026-02-04 14:23:06.134035] Batch processing 769 files...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[2026-02-04 14:26:20.873918] Complete!
Total DOIs Matched:       48,961,051
Total Records Saved:      1,565,793


In [3]:
def count_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        count = 0
        for _ in f:
            count += 1
    return count
file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
print(f"Total objects: {count_lines(file_path):,}")

Total objects: 1,565,793


## Get Datacite citations

### Extract unique DOIs from DataCite citations

In [32]:
# Extract unique DOIs from DataCite citations
folder_path = r"D:\pipeline-data\citations\datacite\citation_blocks"
output_parquet = r"D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet"
extract_unique_dois_from_citation_blocks(folder_path, output_parquet)

[*] Scanning 1 files in citation_blocks...
    -> Finished citation_blocks.ndjson. Current unique DOIs: 896,417

[*] Final Save: Exporting 896,417 unique DOIs to D:\pipeline-data\citations\datacite\datacite_unique_citation_dois...
[SUCCESS] Aggregate DOI list saved to D:\pipeline-data\citations\datacite\datacite_unique_citation_dois


In [4]:
duckdb.execute(f"SELECT* FROM read_parquet('{output_parquet}') LIMIT 10").df()

,doi,pubdate
0,10.1021/acs.inorgchem.6b01545,2016-09-27
1,10.12688/f1000research.2-159.v1,2013-07-17
2,10.1257/aer.20210413,2023-05-31
3,10.1080/2150704x.2016.1234726,2016-09-29
4,10.1159/000540058,2024-06-27
5,10.15468/dl.dr0m6g,2020-03-06
6,10.57451/lhd.a.gas_puf.168994.1,2025-09-10
7,10.15468/dl.dbrhsv,2020-04-10
8,10.1159/000541840,2024-11-13
9,10.5194/os-11-503-2015,2015-07-03


### Find publication dates for these DOIs in the OpenAlex Snapshot

In [1]:
# Paths
input_parquet = r"D:\pipeline-data\citations\datacite\datacite_unique_citation_dois.parquet"
output_parquet = r"D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"
oa_db_path = r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb"

In [15]:
# Run
lookup_dates_in_oa_snapshot(oa_db_path, input_parquet, output_parquet)

Joining D:\pipeline-data\citations\datacite\datacite_unique_citation_dois with OpenAlex database...
    [SUCCESS] Found 884,723 matches.
    [INFO] Results saved to: D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates
    [INFO] Time taken: 41.88s


In [16]:
# View
duckdb.execute(f"SELECT* FROM read_parquet('{output_parquet}') LIMIT 5").df()

,doi,pubdate
0,10.1021/acs.inorgchem.6b01545,2016-09-27
1,10.12688/f1000research.2-159.v1,2013-07-17
2,10.1257/aer.20210413,2023-05-31
3,10.1080/2150704x.2016.1234726,2016-09-29
4,10.1159/000540058,2024-06-27


In [17]:
#Check
con = duckdb.connect(r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb")

# Replace with the DOI you want to check
test_doi = '10.1257/aer.20210413'

# Direct lookup
result = con.execute("SELECT doi, pubdate FROM openalex_pubdate WHERE doi = ?", [test_doi]).df()
print(result)
con.close()

                    doi     pubdate
0  10.1257/aer.20210413  2021-03-01
1  10.1257/aer.20210413  2023-05-31


### Create citations file

In [15]:
folder_path = r"D:\pipeline-data\citations\datacite\citation_blocks"
out_ndjson = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
date_parquet = r"D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"

In [16]:
file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
batch_find_citations_from_dc_parallel(file_path, out_ndjson, date_parquet)

[*] Counting exact lines in input file...
    -> Total workload: 1,565,793 lines.
[*] Using 8 workers (reduced to save RAM)...
[*] Launching workers...
Progress: 0.00% | Processed: 0/1,565,793

TypeError: datacite_citations_block_to_records_unified() got an unexpected keyword argument 'dataset_p'